# Automated Design of Agentic Systems (ADAS) | Frameworks & Meta-Approaches

In [1]:
# ADAS: Prompt Optimization via Automated Design Search
# Meta-agent proposes system prompt variations, each is ACTUALLY TESTED on labeled examples,
# best performer is mutated in the next iteration. Evaluation uses keyword/format heuristics.
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from dataclasses import dataclass
import re

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
# --- Labeled test suite for a sentiment classification task ---
TEST_SUITE = [
    {"input": "This product is amazing, I love it!", "label": "positive"},
    {"input": "Terrible quality, broke after one day.", "label": "negative"},
    {"input": "It works fine, nothing special.", "label": "neutral"},
    {"input": "Best purchase I've ever made!", "label": "positive"},
    {"input": "Waste of money, very disappointed.", "label": "negative"},
]

@dataclass
class CandidateAgent:
    system_prompt: str
    score: float = 0.0

def evaluate_candidate(candidate: CandidateAgent) -> float:
    """Run candidate agent on ALL test cases; score by keyword matching (not LLM-as-judge)."""
    correct = 0
    for tc in TEST_SUITE:
        response = model.invoke([
            {"role": "system", "content": candidate.system_prompt},
            {"role": "user", "content": f"Classify this review: {tc['input']}"},
        ])
        answer = response.content.strip().lower()
        # Heuristic evaluation: check if the correct label appears in the response
        # Also verify format: response should be short (under 50 words) and contain the label
        has_label = tc["label"] in answer
        is_concise = len(answer.split()) < 50
        has_format = bool(re.search(r"(positive|negative|neutral)", answer))
        if has_label and is_concise and has_format:
            correct += 1
    return correct / len(TEST_SUITE)

def generate_initial_prompts() -> list:
    """Meta-agent proposes 3 diverse system prompt variations."""
    response = model.invoke(
        "Generate 3 different system prompts for a sentiment classifier agent.\n"
        "Each should instruct the agent to classify text as positive/negative/neutral.\n"
        "Vary the style: one terse, one detailed, one chain-of-thought.\n"
        "Return EXACTLY 3 prompts separated by '---'."
    )
    return [p.strip() for p in response.content.split("---") if p.strip()][:3]

def mutate_prompt(best_prompt: str, best_score: float) -> list:
    """Meta-agent creates 3 mutations of the best-performing prompt."""
    response = model.invoke(
        f"This system prompt scored {best_score:.0%} on sentiment classification:\n"
        f'"""{best_prompt}"""\n\n'
        f"Generate 3 improved variations. Keep what works, fix what doesn't.\n"
        f"Return EXACTLY 3 prompts separated by '---'."
    )
    return [p.strip() for p in response.content.split("---") if p.strip()][:3]

In [4]:
# --- ADAS Search: 2 iterations ---
print("=" * 60)
print("ADAS: Searching for optimal sentiment classification prompt")
print("=" * 60)

# Iteration 1: Evaluate initial candidates
print("\n--- Iteration 1: Initial candidates ---")
prompts = generate_initial_prompts()
candidates = [CandidateAgent(system_prompt=p) for p in prompts]
for i, c in enumerate(candidates):
    c.score = evaluate_candidate(c)
    print(f"  Candidate {i+1}: {c.score:.0%} accuracy | '{c.system_prompt[:60]}...'")
best = max(candidates, key=lambda c: c.score)
print(f"  Winner: {best.score:.0%}")

# Iteration 2: Mutate the winner
print("\n--- Iteration 2: Mutations of best prompt ---")
mutated_prompts = mutate_prompt(best.system_prompt, best.score)
mutants = [CandidateAgent(system_prompt=p) for p in mutated_prompts]
all_candidates = [best] + mutants  # Keep the previous winner in the pool
for i, c in enumerate(mutants):
    c.score = evaluate_candidate(c)
    print(f"  Mutant {i+1}: {c.score:.0%} accuracy | '{c.system_prompt[:60]}...'")
final_best = max(all_candidates, key=lambda c: c.score)

improvement = final_best.score - best.score
print(f"\nMutations applied: {len(mutants)} | Best improvement: {improvement:+.0%} accuracy")

print(f"\n{'=' * 60}")
print(f"FINAL BEST: {final_best.score:.0%} accuracy")
print(f"Prompt: {final_best.system_prompt[:200]}")
print(f"{'=' * 60}")

ADAS: Searching for optimal sentiment classification prompt

--- Iteration 1: Initial candidates ---
  Candidate 1: 100% accuracy | 'Classify sentiment: positive, negative, or neutral....'
  Candidate 2: 100% accuracy | 'You are tasked with analyzing the sentiment of textual data ...'
  Candidate 3: 100% accuracy | 'Consider the text in question. Begin by identifying any over...'
  Winner: 100%

--- Iteration 2: Mutations of best prompt ---
  Mutant 1: 100% accuracy | 'Classify the sentiment of the text as positive, negative, or...'
  Mutant 2: 100% accuracy | 'Determine whether the sentiment expressed is positive, negat...'
  Mutant 3: 100% accuracy | 'Identify the sentiment conveyed: positive, negative, or neut...'

Mutations applied: 3 | Best improvement: +0% accuracy

FINAL BEST: 100% accuracy
Prompt: Classify sentiment: positive, negative, or neutral.
